In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000000/00000000.arcs
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000000/00000000.nodes
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000000/00000000.route
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000000/00000000.track
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000007/00000007.arcs
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000007/00000007.nodes
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000007/00000007.route
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000007/00000007.track
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000009/00000009.arcs
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000009/00000009.nodes
/ka

In [7]:
import pandas as pd
import numpy as np
import networkx as nx
import os
import joblib
import time
import glob

# Paths updated to match your dataset7
MODEL_PATH = "/kaggle/input/datasets/shristisaha903/dataset7/logistic_regression_tuned_v2.pkl" 
SCALER_PATH = "/kaggle/input/datasets/shristisaha903/dataset7/scaler_v2.pkl"

print("Loading tuned model and scaler...")
model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
print("Models loaded successfully!")

Loading tuned model and scaler...
Models loaded successfully!


In [20]:
def viterbi_decode(trip_df, G, transition_weight=0.5):
    points = sorted(trip_df['point_idx'].unique())
    
    V = [{}] 
    path = {}
    
    # --- INITIALIZATION ---
    first_point = points[0]
    first_cands = trip_df[trip_df['point_idx'] == first_point]
    for _, row in first_cands.iterrows():
        arc = row['arc_id']
        V[0][arc] = np.log(row['prob'] + 1e-9)
        path[arc] = [arc]
        
    # --- RECURSION ---
    for t in range(1, len(points)):
        V.append({})
        new_path = {}
        
        curr_point = points[t]
        prev_point = points[t-1]
        
        curr_cands = trip_df[trip_df['point_idx'] == curr_point]
        prev_cands = trip_df[trip_df['point_idx'] == prev_point]
        
        for _, c_row in curr_cands.iterrows():
            c_arc = c_row['arc_id']
            max_prob = -np.inf
            best_prev = None
            
            for _, p_row in prev_cands.iterrows():
                p_arc = p_row['arc_id']
                
                # 1. Emission Prob
                emission = np.log(c_row['prob'] + 1e-9)
                
                # 2. Transition Prob
                if p_arc == c_arc or G.has_edge(p_arc, c_arc):
                    trans = 0.0
                else:
                    # The softer penalty that gave 89.26%
                    trans = np.log(0.1) * transition_weight 
                
                # Viterbi Recurrence
                prob = V[t-1][p_arc] + trans + emission
                
                if prob > max_prob:
                    max_prob = prob
                    best_prev = p_arc
                    
            V[t][c_arc] = max_prob
            if best_prev is not None:
                new_path[c_arc] = path[best_prev] + [c_arc]
            else:
                new_path[c_arc] = [c_arc]
                
        path = new_path
        
    # --- TERMINATION ---
    best_final_arc = max(V[-1], key=V[-1].get)
    return path[best_final_arc]

print("Reverted Viterbi function defined.")

Reverted Viterbi function defined.


In [21]:
print("Loading validation data...")
DATA_PATH = "/kaggle/input/datasets/shristisaha903/dataset6/advanced_features_full_v2.csv" 
df = pd.read_csv(DATA_PATH)

from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, val_idx = next(gss.split(df, df['target'], groups=df['trip_id']))
val_df_full = df.iloc[val_idx]

RAW_DATA_DIR = "/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/train"
val_trips = val_df_full['trip_id'].unique()[:5]

classifier_accs = []
hmm_accs = []

exclude_cols = ['target', 'point_idx', 'trip_id', 'arc_id']
features = [c for c in df.columns if c not in exclude_cols]

for trip_id in val_trips:
    print(f"\nProcessing Trip {trip_id}...")
    trip_df = val_df_full[val_df_full['trip_id'] == trip_id].copy()
    
    X_test = trip_df[features]
    X_test_scaled = scaler.transform(X_test)
    trip_df['prob'] = model.predict_proba(X_test_scaled)[:, 1]
    
    # Classifier-Only
    clf_preds = trip_df.loc[trip_df.groupby("point_idx")["prob"].idxmax()]
    clf_acc = (clf_preds['target'] == 1).mean() 
    classifier_accs.append(clf_acc)
    
    # Build Graph
    trip_id_str = str(trip_id).zfill(8)
    stem = os.path.join(RAW_DATA_DIR, trip_id_str, trip_id_str)
    if not os.path.exists(stem + ".arcs"):
        continue
        
    arcs = pd.read_csv(stem + ".arcs", sep=r"\s+", header=None, names=["from_node", "to_node"])
    arcs["arc_id"] = arcs.index
    
    G = nx.DiGraph()
    node_to_arcs = {}
    for _, row in arcs.iterrows():
        G.add_edge(row['arc_id'], row['arc_id'])
        if row['from_node'] not in node_to_arcs:
            node_to_arcs[row['from_node']] = []
        node_to_arcs[row['from_node']].append(row['arc_id'])
        
    for _, row in arcs.iterrows():
        to_node = row['to_node']
        if to_node in node_to_arcs:
            for next_arc in node_to_arcs[to_node]:
                G.add_edge(row['arc_id'], next_arc)
                
    # HMM Decoding
    start_hmm = time.time()
    predicted_route = viterbi_decode(trip_df, G, transition_weight=0.5)
    
    hmm_correct = 0
    for i, pred_arc in enumerate(predicted_route):
        point_data = trip_df[trip_df['point_idx'] == i]
        if not point_data.empty and point_data['target'].max() == 1:
            true_arc = point_data[point_data['target'] == 1]['arc_id'].values[0]
            if pred_arc == true_arc:
                hmm_correct += 1
                
    hmm_acc = hmm_correct / len(predicted_route)
    hmm_accs.append(hmm_acc)
    
    print(f"  Classifier-Only Acc: {clf_acc:.4f}")
    print(f"  HMM + Viterbi Acc:   {hmm_acc:.4f}")
    print(f"  (Decoded {len(predicted_route)} points in {time.time()-start_hmm:.2f}s)")

print("\n=== FINAL ROUTE RECONSTRUCTION RESULTS ===")
print(f"Average Classifier-Only Accuracy: {np.mean(classifier_accs):.4f}")
print(f"Average HMM + Viterbi Accuracy:   {np.mean(hmm_accs):.4f}")

Loading validation data...

Processing Trip 1...
  Classifier-Only Acc: 0.9316
  HMM + Viterbi Acc:   0.9247
  (Decoded 1315 points in 2.61s)

Processing Trip 5...
  Classifier-Only Acc: 0.9856
  HMM + Viterbi Acc:   0.9780
  (Decoded 1045 points in 2.07s)

Processing Trip 6...
  Classifier-Only Acc: 0.8952
  HMM + Viterbi Acc:   0.6683
  (Decoded 1450 points in 3.12s)

Processing Trip 12...
  Classifier-Only Acc: 0.9739
  HMM + Viterbi Acc:   0.9708
  (Decoded 1608 points in 3.26s)

Processing Trip 13...
  Classifier-Only Acc: 0.9081
  HMM + Viterbi Acc:   0.9213
  (Decoded 2580 points in 5.42s)

=== FINAL ROUTE RECONSTRUCTION RESULTS ===
Average Classifier-Only Accuracy: 0.9389
Average HMM + Viterbi Accuracy:   0.8926


In [26]:
import os
import json
import pandas as pd
import numpy as np
import networkx as nx
import joblib

# 1. SETUP PATHS
BASE_DIR = "/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test"
OUTPUT_DIR = "/kaggle/working/api_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = "/kaggle/input/datasets/shristisaha903/dataset7/logistic_regression_tuned_v2.pkl"
SCALER_PATH = "/kaggle/input/datasets/shristisaha903/dataset7/scaler_v2.pkl"
model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

# --- FIX: Load exact feature names from the training CSV ---
DATA_PATH = "/kaggle/input/datasets/shristisaha903/dataset6/advanced_features_full_v2.csv"
train_df_cols = pd.read_csv(DATA_PATH, nrows=1).columns
exclude_cols = ['target', 'point_idx', 'trip_id', 'arc_id']
train_features = [c for c in train_df_cols if c not in exclude_cols]
print(f"Using {len(train_features)} features for inference: {train_features}")
# -----------------------------------------------------------

# 3. HELPER FUNCTIONS
def load_track(path):
    return pd.read_csv(path, sep=r"\s+", header=None, names=["lon", "lat", "t"])

def load_nodes(path):
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["lon", "lat"])
    df["node_id"] = df.index
    return df

def load_arcs(path):
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["from_node", "to_node"])
    df["arc_id"] = df.index
    return df

def get_candidates_and_meta(tr, nodes, arcs, max_candidates=3):
    lon0, lat0 = tr.lon.mean(), tr.lat.mean()
    px = (tr.lon.values - lon0) * 111320 * np.cos(np.radians(lat0))
    py = (tr.lat.values - lat0) * 111320
    nx_coords = (nodes.lon.values - lon0) * 111320 * np.cos(np.radians(lat0))
    ny_coords = (nodes.lat.values - lat0) * 111320
    
    from_x, from_y = nx_coords[arcs.from_node.values], ny_coords[arcs.from_node.values]
    to_x, to_y = nx_coords[arcs.to_node.values], ny_coords[arcs.to_node.values]
    
    abx, aby = to_x - from_x, to_y - from_y
    ab2 = abx**2 + aby**2
    ab2[ab2 == 0] = 1e-9 
    
    arc_lengths = np.sqrt(abx**2 + aby**2)
    arc_meta = pd.DataFrame({"arc_id": arcs.arc_id, "arc_length": arc_lengths})
    
    candidates = []
    for i in range(len(px)):
        apx, apy = px[i] - from_x, py[i] - from_y
        t = np.clip((apx * abx + apy * aby) / ab2, 0.0, 1.0)
        cx, cy = from_x + t * abx, from_y + t * aby
        dists = np.sqrt((px[i] - cx)**2 + (py[i] - cy)**2)
        
        closest_indices = np.argpartition(dists, max_candidates)[:max_candidates]
        closest_indices = closest_indices[np.argsort(dists[closest_indices])]
        
        for rank, idx in enumerate(closest_indices):
            dist_to_start = np.sqrt((px[i]-from_x[idx])**2 + (py[i]-from_y[idx])**2)
            dist_to_end = np.sqrt((px[i]-to_x[idx])**2 + (py[i]-to_y[idx])**2)
            
            candidates.append({
                'point_idx': i,
                'arc_id': arcs.arc_id.iloc[idx],
                'distance_m': dists[idx],
                'rank': rank,
                'dist_to_start_node': dist_to_start,
                'dist_to_end_node': dist_to_end
            })
    return pd.DataFrame(candidates), arc_meta

# 4. PROCESS ONE TEST TRIP
def process_test_trip(trip_id):
    stem = os.path.join(BASE_DIR, trip_id, trip_id)
    tr = load_track(stem + ".track")
    nodes = load_nodes(stem + ".nodes")
    arcs = load_arcs(stem + ".arcs")
    
    # Feature Engineering
    lon0, lat0 = tr.lon.mean(), tr.lat.mean()
    tr["x"] = (tr.lon.values - lon0) * 111320 * np.cos(np.radians(lat0))
    tr["y"] = (tr.lat.values - lat0) * 111320
    tr["dt"] = tr["t"].diff().fillna(1).clip(0, 3600)
    tr["dx"] = tr["x"].diff().fillna(0)
    tr["dy"] = tr["y"].diff().fillna(0)
    tr["delta_dist"] = np.sqrt(tr["dx"]**2 + tr["dy"]**2)
    tr["speed"] = (tr["delta_dist"] / tr["dt"].replace(0, np.nan)).fillna(0).clip(0, 50)
    tr["acceleration"] = tr["speed"].diff().fillna(0)
    tr["heading"] = (np.degrees(np.arctan2(tr["dx"], tr["dy"])) % 360).fillna(0)
    tr["heading_change"] = tr["heading"].diff().fillna(0)
    tr["heading_change"] = tr["heading_change"].apply(lambda x: min(x, 360 - x) if x > 180 else min(x, -360 - x) if x < -180 else x)
    tr["rolling_speed_mean"] = tr["speed"].rolling(window=5, min_periods=1).mean()
    tr["rolling_speed_std"] = tr["speed"].rolling(window=5, min_periods=1).std().fillna(0)
    tr["point_idx"] = tr.index
    tr["time_in_trip"] = tr["t"] - tr["t"].iloc[0]
    tr["time_gap"] = tr["dt"]

    cand, arc_meta = get_candidates_and_meta(tr, nodes, arcs)
    nx_coords = (nodes.lon.values - lon0) * 111320 * np.cos(np.radians(lat0))
    ny_coords = (nodes.lat.values - lat0) * 111320
    from_x, from_y = nx_coords[arcs.from_node.values], ny_coords[arcs.from_node.values]
    to_x, to_y = nx_coords[arcs.to_node.values], ny_coords[arcs.to_node.values]
    road_heading = (np.degrees(np.arctan2(to_x - from_x, to_y - from_y)) % 360)
    arc_meta["road_heading"] = road_heading
    
    G_net = nx.DiGraph()
    G_net.add_edges_from(zip(arcs.from_node, arcs.to_node))
    arc_meta["arc_degree"] = arcs.apply(lambda row: G_net.degree(row.from_node) + G_net.degree(row.to_node), axis=1)

    cand = cand.merge(arc_meta, on="arc_id", how="left")
    cand = cand.merge(tr[["point_idx", "speed", "acceleration", "heading", "heading_change", "time_gap", "rolling_speed_mean", "rolling_speed_std", "time_in_trip"]], on="point_idx", how="left")

    cand["heading_diff"] = np.abs(cand["heading"] - cand["road_heading"])
    cand["heading_diff"] = cand["heading_diff"].apply(lambda x: min(x, 360 - x))
    cand["dist_rank_0"] = cand.groupby("point_idx")["distance_m"].transform("min")
    cand["dist_rank_1"] = cand.groupby("point_idx")["distance_m"].transform(lambda x: x.nsmallest(2).max())
    cand["candidate_spread"] = cand["dist_rank_1"] - cand["dist_rank_0"]
    cand["implied_speed"] = cand["arc_length"] / cand["time_gap"].replace(0, 1)
    cand["next_arc_heading"] = cand.groupby("point_idx")["road_heading"].shift(-1).fillna(cand["road_heading"])
    cand["arc_turn_angle"] = np.abs(cand["road_heading"] - cand["next_arc_heading"])
    cand["arc_turn_angle"] = cand["arc_turn_angle"].apply(lambda x: min(x, 360 - x))
    cand = cand.sort_values(by=["point_idx", "rank"])
    cand["prev_dist"] = cand.groupby("point_idx")["distance_m"].transform("first").shift(1).fillna(0)
    cand["prev_speed"] = cand.groupby("point_idx")["speed"].transform("first").shift(1).fillna(0)
    cand["time_since_prev"] = cand.groupby("point_idx")["time_gap"].transform("first").shift(1).fillna(0)
    
    # Recreate sin/cos features
    cand["heading_sin"] = np.sin(np.radians(cand["heading"]))
    cand["heading_cos"] = np.cos(np.radians(cand["heading"]))
    cand["road_heading_sin"] = np.sin(np.radians(cand["road_heading"]))
    cand["road_heading_cos"] = np.cos(np.radians(cand["road_heading"]))
    
    # Predict using the EXACT feature list
    X_test = cand[train_features]
    X_test_scaled = scaler.transform(X_test)
    cand['prob'] = model.predict_proba(X_test_scaled)[:, 1]
    
    # Build HMM Graph
    G_hmm = nx.DiGraph()
    node_to_arcs = {}
    for _, row in arcs.iterrows():
        G_hmm.add_edge(row['arc_id'], row['arc_id'])
        if row['from_node'] not in node_to_arcs:
            node_to_arcs[row['from_node']] = []
        node_to_arcs[row['from_node']].append(row['arc_id'])
    for _, row in arcs.iterrows():
        to_node = row['to_node']
        if to_node in node_to_arcs:
            for next_arc in node_to_arcs[to_node]:
                G_hmm.add_edge(row['arc_id'], next_arc)
                
    # Run Viterbi
    predicted_route = viterbi_decode(cand, G_hmm, transition_weight=0.5)
    
    # Save JSON
    candidates_data = cand[['point_idx', 'arc_id', 'prob']].to_dict(orient="records")
    graph_data = {
        "nodes": nodes.to_dict(orient="records"),
        "arcs": arcs.to_dict(orient="records")
    }
    route_data = [int(arc) for arc in predicted_route]
    
    output_json = {
        "trip_id": trip_id,
        "candidates": candidates_data,
        "graph": graph_data,
        "predicted_route": route_data
    }
    
    with open(os.path.join(OUTPUT_DIR, f"{trip_id}.json"), "w") as f:
        json.dump(output_json, f)

# 5. RUN FOR ALL 10 TEST TRIPS
test_trip_ids = ["00000000", "00000007", "00000009", "00000040", "00000052", 
                 "00000056", "00000065", "00000076", "00000077", "00000091"]

print(f"Generating JSONs for {len(test_trip_ids)} student test trips...")
for trip_id in test_trip_ids:
    print(f"Processing Trip {trip_id}...")
    process_test_trip(trip_id)
    
print(f"\nSuccessfully generated JSONs in {OUTPUT_DIR}")
print("Download the 'api_data' folder from the Output tab.")

Using 25 features for inference: ['distance_m', 'rank', 'dist_to_start_node', 'dist_to_end_node', 'arc_length', 'road_heading', 'arc_degree', 'speed', 'acceleration', 'heading', 'heading_change', 'time_gap', 'rolling_speed_mean', 'rolling_speed_std', 'time_in_trip', 'heading_diff', 'dist_rank_0', 'dist_rank_1', 'candidate_spread', 'implied_speed', 'next_arc_heading', 'arc_turn_angle', 'prev_dist', 'prev_speed', 'time_since_prev']
Generating JSONs for 10 student test trips...
Processing Trip 00000000...
Processing Trip 00000007...
Processing Trip 00000009...
Processing Trip 00000040...
Processing Trip 00000052...
Processing Trip 00000056...
Processing Trip 00000065...
Processing Trip 00000076...
Processing Trip 00000077...
Processing Trip 00000091...

Successfully generated JSONs in /kaggle/working/api_data
Download the 'api_data' folder from the Output tab.
